# **Queries to test on AI Agent.**

## **Connecting to Database**

In [11]:
import os

print(os.listdir())

['.config', 'pockettoons.db', '.ipynb_checkpoints', 'sample_data']


In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("pockettoons.db")

print("✅ Connected!")

✅ Connected!


In [13]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,users
1,sessions
2,transactions
3,content_views


In [14]:
for table in ["users", "sessions", "transactions", "content_views"]:
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 3", conn)
    print(f"\n📋 {table.upper()}")
    print(df.to_string())
    print("-"*60)


📋 USERS
   user_id country platform plan_type signup_date age_bucket gender  is_active
0        1      IN      web      free  2024-12-18      13-17  other          1
1        2      IN  android      free  2024-12-07        45+  other          0
2        3      IN      web      free  2024-10-25      25-34      M          1
------------------------------------------------------------

📋 SESSIONS
   session_id  user_id platform               session_start                 session_end  duration_seconds country
0           1     6253  android  2024-11-04T11:21:35.100378  2024-11-04T11:36:42.100378               907      IN
1           2     4615  android  2024-12-07T00:27:12.783430  2024-12-07T00:41:30.783430               858      CA
2           3     9914      ios  2024-12-11T17:54:37.105954  2024-12-11T18:03:37.105954               540      CA
------------------------------------------------------------

📋 TRANSACTIONS
   transaction_id  user_id                   timestamp  amount_usd pl



# **SECTION 1 - Basic Product Metrics**

**QUERY 1 — Total number of Users**

In [15]:
query = """
SELECT
    COUNT(*) AS total_users
FROM users;
"""

pd.read_sql(query, conn)

,total_users
0,10000


 **QUERY 2 — Calculating DAU (Daily Active Users)**

In [16]:
query = """
SELECT
    DATE(session_start) AS session_day,
    COUNT(DISTINCT user_id) AS dau
FROM sessions
GROUP BY session_day
ORDER BY session_day;
"""

pd.read_sql(query, conn)

,session_day,dau
0,2024-10-01,188
1,2024-10-02,365
2,2024-10-03,401
3,2024-10-04,396
4,2024-10-05,394
...,...,...
118,2025-01-27,404
119,2025-01-28,384
120,2025-01-29,378
121,2025-01-30,393


 **QUERY 3 — Calculating MAU (Monthly Active Users)**

In [17]:

query = """
    SELECT
        strftime('%Y-%m', session_start) AS month,
        COUNT(DISTINCT user_id) AS mau
    FROM sessions
    GROUP BY month
    ORDER BY month;
"""
pd.read_sql(query, conn)

,month,mau
0,2024-10,7094
1,2024-11,7073
2,2024-12,7177
3,2025-01,7153


**QUERY 4 — Users by Platform**
Can be Used for customer Segmentation

1.   Can be Used for customer Segmentation




In [18]:
query = """
SELECT
    platform,
    COUNT(*) AS total_users
FROM users
GROUP BY platform
ORDER BY total_users DESC;
"""
pd.read_sql(query, conn)

,platform,total_users
0,android,3407
1,ios,3326
2,web,3267


# **SECTION 2 - Revenue Analytics**

**QUERY 5 — Total revenue generated**

In [19]:
query = """
SELECT
    ROUND(SUM(amount_usd), 2)
    AS total_revenue
FROM transactions

WHERE status = 'success';
"""

pd.read_sql(query, conn)

,total_revenue
0,180658.77


**QUERY 6 — Revenue by Country**


1.   Helps us to understand which country generating most revenue and comparing wiht others




In [20]:
query = """
SELECT
    users.country,
    ROUND(SUM(transactions.amount_usd), 2)
    AS revenue

FROM transactions

JOIN users
ON transactions.user_id = users.user_id

WHERE transactions.status = 'success'

GROUP BY users.country

ORDER BY revenue DESC;
"""

pd.read_sql(query, conn)

,country,revenue
0,IN,82015.50
1,US,35615.22
2,GB,15373.33
3,CA,9626.83
4,OTHER,8602.86
5,AU,8449.05
6,SG,7824.20
7,DE,6785.39
8,AE,6366.39


**QUERY 7 — Revenue by Plan Type**

In [21]:
query = """
SELECT
    plan_type,
    ROUND(SUM(amount_usd), 2) AS revenue

FROM transactions

WHERE status = 'success'

GROUP BY plan_type;
"""
pd.read_sql(query, conn)

,plan_type,revenue
0,basic,70179.36
1,premium,110479.41


# **SECTION 3 - Engagement Metrics**

**QUERY 8 — Average Session Duration**

In [22]:
query = """
SELECT
    ROUND(AVG(duration_seconds)/60, 2)
    AS avg_session_minutes
FROM sessions;
"""
pd.read_sql(query, conn)

,avg_session_minutes
0,9.92


**QUERY 9 — Top Genres by Views**

In [23]:
query = """
SELECT
    genre,
    COUNT(*) AS total_views

FROM content_views

GROUP BY genre

ORDER BY total_views DESC;
"""
pd.read_sql(query, conn)

,genre,total_views
0,action,4595
1,horror,4556
2,thriller,4487
3,sci-fi,4472
4,romance,4470
5,comedy,4428
6,slice_of_life,2992


**QUERY 10 — Top Content Titles**

In [24]:
query = """
SELECT
    content_title,
    COUNT(*) AS total_views

FROM content_views

GROUP BY content_title

ORDER BY total_views DESC

LIMIT 10;
"""
pd.read_sql(query, conn)

,content_title,total_views
0,Blood Pact,1588
1,Office War,1554
2,The Fake Marriage,1533
3,Dark Whispers,1520
4,Underworld Kiss,1519
5,Steel Hearts,1511
6,Cherry Blossom Trap,1506
7,Lovesick CEO,1503
8,Crimson Thread,1501
9,Shadow Hunter,1498


# **Section 4 → Filtered Aggregates**


**Query 11 - Calculating DAU(Daily Active Users) Specially for US.**


1.   Can be used for filtering based on Country.


In [25]:

query = """
    SELECT
        DATE(session_start) AS date,
        COUNT(DISTINCT user_id) AS us_dau
    FROM sessions
    WHERE country = 'US'
    GROUP BY DATE(session_start)
    ORDER BY date;
"""

pd.read_sql(query, conn)

,date,us_dau
0,2024-10-01,34
1,2024-10-02,69
2,2024-10-03,81
3,2024-10-04,71
4,2024-10-05,73
...,...,...
118,2025-01-27,71
119,2025-01-28,70
120,2025-01-29,86
121,2025-01-30,90


**Query 12 - Calculating premium users do we have in india**


In [26]:
query = """
SELECT
    COUNT(user_id) AS premium_users_in_india
FROM users
WHERE plan_type = 'premium' AND country = 'IN';
"""

pd.read_sql(query, conn)

,premium_users_in_india
0,886


**Query 13 - Premium Vs Free Users**

In [27]:
query = """
SELECT
    plan_type,
    COUNT(*) AS users_count

FROM users

GROUP BY plan_type;
"""
pd.read_sql(query, conn)

,plan_type,users_count
0,basic,2540
1,free,5451
2,premium,2009


# **Section 5 → Cohort & Retention**



**Query 14 - D7 Retention by Cohort.**

In [28]:
query = """
WITH cohort AS (
    SELECT
        user_id,
        signup_date,
        strftime('%Y-%m', signup_date) AS cohort_month
    FROM users
),
retained AS (
    SELECT DISTINCT
        c.user_id,
        c.cohort_month
    FROM cohort c
    JOIN sessions s ON s.user_id = c.user_id
    WHERE DATE(s.session_start)
          BETWEEN c.signup_date
          AND DATE(c.signup_date, '+7 days')
)
SELECT
    c.cohort_month,
    COUNT(DISTINCT c.user_id)    AS cohort_size,
    COUNT(DISTINCT r.user_id)    AS retained_d7,
    ROUND(
        100.0 * COUNT(DISTINCT r.user_id)
        / COUNT(DISTINCT c.user_id), 1
    ) AS d7_retention_pct
FROM cohort c
LEFT JOIN retained r ON r.user_id = c.user_id
GROUP BY c.cohort_month
ORDER BY c.cohort_month
"""
pd.read_sql(query, conn)

,cohort_month,cohort_size,retained_d7,d7_retention_pct
0,2024-10,2522,693,27.5
1,2024-11,2482,697,28.1
2,2024-12,2481,639,25.8
3,2025-01,2515,594,23.6


**Query 15 - Premium vs Free D7**      

In [29]:
query = """
WITH cohort AS (
    SELECT user_id, signup_date, plan_type
    FROM users
),
retained AS (
    SELECT DISTINCT c.user_id
    FROM cohort c
    JOIN sessions s ON s.user_id = c.user_id
    WHERE DATE(s.session_start)
          BETWEEN c.signup_date
          AND DATE(c.signup_date, '+7 days')
)
SELECT
    c.plan_type,
    COUNT(DISTINCT c.user_id)    AS cohort_size,
    COUNT(DISTINCT r.user_id)    AS retained_d7,
    ROUND(
        100.0 * COUNT(DISTINCT r.user_id)
        / COUNT(DISTINCT c.user_id), 1
    ) AS d7_retention_pct
FROM cohort c
LEFT JOIN retained r ON r.user_id = c.user_id
GROUP BY c.plan_type
ORDER BY d7_retention_pct DESC
"""
pd.read_sql(query, conn)

,plan_type,cohort_size,retained_d7,d7_retention_pct
0,basic,2540,699,27.5
1,premium,2009,522,26.0
2,free,5451,1402,25.7


# **Section 6 → Comparisons**


**Query 16 - This Week vs Last Week**

In [31]:
query = """
SELECT
    SUM(CASE
        WHEN DATE(session_start)
        BETWEEN '2025-01-18' AND '2025-01-24'
        THEN 1 ELSE 0
    END) AS last_week_sessions,

    SUM(CASE
        WHEN DATE(session_start)
        BETWEEN '2025-01-25' AND '2025-01-31'
        THEN 1 ELSE 0
    END) AS this_week_sessions,

    ROUND(
        100.0 * (
            SUM(CASE WHEN DATE(session_start)
                BETWEEN '2025-01-25' AND '2025-01-31'
                THEN 1 ELSE 0 END) -
            SUM(CASE WHEN DATE(session_start)
                BETWEEN '2025-01-18' AND '2025-01-24'
                THEN 1 ELSE 0 END)
        ) / SUM(CASE WHEN DATE(session_start)
            BETWEEN '2025-01-18' AND '2025-01-24'
            THEN 1 ELSE 0 END)
    , 1) AS pct_change
FROM sessions
"""
pd.read_sql(query, conn)

,last_week_sessions,this_week_sessions,pct_change
0,2845,2657,-6.6
